# 01 — Exploratory Data Analysis (EDA)
## BAMIS Fraud Detection | DATATHON ESP DATACLUB 2026

### Objectif
Comprendre la structure et la qualité des données avant toute modélisation.

### Plan
| Section | Contenu |
|---|---|
| 1 | Chargement & aperçu global |
| 2 | Qualité des données (nulls, types, doublons) |
| 3 | Distribution des statuts et services |
| 4 | Analyse des montants |
| 5 | Analyse temporelle (heure, jour, mois) |
| 6 | Analyse des acteurs (expéditeurs / destinataires) |
| 7 | Détection préliminaire d'anomalies |
| 8 | Synthèse & recommandations feature engineering |


In [ ]:
import sys, warnings, os
sys.path.insert(0, '..')
warnings.filterwarnings('ignore')

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
import seaborn as sns
from pathlib import Path

sns.set_theme(style='darkgrid', palette='muted', font_scale=1.1)
plt.rcParams['figure.dpi'] = 110
pd.set_option('display.max_columns', 30)
pd.set_option('display.float_format', '{:,.2f}'.format)

os.makedirs('../reports/figures', exist_ok=True)

print("✅ Imports OK")
print(f"   pandas  {pd.__version__}")
print(f"   numpy   {np.__version__}")


---
## 1. Chargement & Aperçu Global

Le fichier brut fait **368 Mo / ~1,6 million de lignes**.  
On charge un échantillon de **150 000 lignes** pour l'EDA interactive.


In [ ]:
DATA_PATH = Path('../src/data/DATASET_ESP-2026.csv')
SAMPLE_ROWS = 150_000

print(f"Taille fichier : {DATA_PATH.stat().st_size / 1e6:.1f} Mo")

# Lecture brute pour compter les lignes sans tout charger en mémoire
with open(DATA_PATH, 'r', encoding='utf-8', errors='replace') as f:
    total_lines = sum(1 for _ in f)
print(f"Lignes totales : {total_lines - 1:,}  (hors header)")

# Chargement échantillon
df_raw = pd.read_csv(
    DATA_PATH,
    dtype=str,
    on_bad_lines='warn',
    low_memory=False,
    nrows=SAMPLE_ROWS,
    encoding='utf-8',
    encoding_errors='replace',
)
df_raw.columns = df_raw.columns.str.strip().str.upper()
print(f"\nÉchantillon chargé : {df_raw.shape[0]:,} lignes × {df_raw.shape[1]} colonnes")


In [ ]:
# Aperçu des premières lignes
print("── Colonnes disponibles ──")
for i, col in enumerate(df_raw.columns, 1):
    print(f"  {i:2}. {col}")


In [ ]:
df_raw.head(5)


---
## 2. Qualité des Données


In [ ]:
# ── 2a. Valeurs nulles / vides ────────────────────────────────────────────
df_check = df_raw.copy()
# Remplacer chaînes vides par NaN
df_check = df_check.replace(r'^\s*$', np.nan, regex=True)

null_pct = df_check.isnull().mean().sort_values(ascending=False) * 100
null_df = pd.DataFrame({'null_%': null_pct.round(2), 'null_count': df_check.isnull().sum()})
print("Colonnes avec valeurs manquantes :")
print(null_df[null_df['null_%'] > 0].to_string())


In [ ]:
# ── 2b. Heatmap des valeurs nulles ────────────────────────────────────────
cols_with_nulls = null_df[null_df['null_%'] > 0].index.tolist()

fig, ax = plt.subplots(figsize=(14, 4))
null_matrix = df_check[cols_with_nulls].isnull().astype(int)
# Échantillon pour heatmap lisible
sample_idx = np.linspace(0, len(null_matrix)-1, min(500, len(null_matrix)), dtype=int)
sns.heatmap(null_matrix.iloc[sample_idx].T, ax=ax,
            cmap='YlOrRd', cbar=True, xticklabels=False, linewidths=0)
ax.set_title('Carte des valeurs manquantes (500 lignes échantillonnées)', fontweight='bold')
ax.set_xlabel('Transactions (échantillon)')
plt.tight_layout()
plt.savefig('../reports/figures/01_missing_values.png', bbox_inches='tight')
plt.show()


In [ ]:
# ── 2c. Doublons ──────────────────────────────────────────────────────────
dup_tx = df_check['TRANSACTION_CODE'].duplicated(keep=False).sum()
print(f"TRANSACTION_CODE doublons : {dup_tx:,}  ({dup_tx/len(df_check)*100:.3f}%)")

# Types effectifs après nettoyage
print("\n── Exemple valeurs par colonne ──")
for col in ['TRANSACTION_STATUS', 'SERVICE_CODE', 'CHANNEL_TYPE', 'DESTINATION_TYPE']:
    vals = df_check[col].dropna().unique()[:6]
    print(f"  {col:<30} : {list(vals)}")


---
## 3. Distribution des Statuts et Services


In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(18, 5))
fig.suptitle('Distribution des catégories principales', fontsize=14, fontweight='bold')

# Transaction status
status_counts = df_check['TRANSACTION_STATUS'].value_counts()
axes[0].barh(status_counts.index, status_counts.values,
             color=sns.color_palette('muted', len(status_counts)))
axes[0].set_title('Statut des transactions')
axes[0].set_xlabel('Nombre')
for i, v in enumerate(status_counts.values):
    axes[0].text(v + 50, i, f'{v:,}', va='center', fontsize=9)

# Service code
svc_counts = df_check['SERVICE_CODE'].value_counts().head(10)
axes[1].barh(svc_counts.index, svc_counts.values,
             color=sns.color_palette('coolwarm', len(svc_counts)))
axes[1].set_title('Top 10 Service Codes')
axes[1].set_xlabel('Nombre')

# Channel type
ch_counts = df_check['CHANNEL_TYPE'].value_counts().dropna()
if len(ch_counts):
    axes[2].pie(ch_counts.values, labels=ch_counts.index,
                autopct='%1.1f%%', startangle=90,
                colors=sns.color_palette('pastel', len(ch_counts)))
    axes[2].set_title('Canal de transaction')
else:
    axes[2].text(0.5, 0.5, 'Données non disponibles', ha='center', va='center')
    axes[2].set_title('Canal de transaction')

plt.tight_layout()
plt.savefig('../reports/figures/01_categories.png', bbox_inches='tight')
plt.show()

print("\n── Résumé statuts ──")
for s, c in status_counts.items():
    print(f"  {s:<25} {c:>8,}  ({c/len(df_check)*100:.2f}%)")


---
## 4. Analyse des Montants

Les montants sont stockés en **sous-unités XOF** (diviser par 1 000 000 pour obtenir des XOF).


In [ ]:
# Conversion montants
df_check['AMOUNT_XOF'] = pd.to_numeric(
    df_check['TRANSACTION_AMOUNT'].str.strip().str.replace(r'[^\d.\-]','',regex=True),
    errors='coerce'
)
df_check['FEES_XOF'] = pd.to_numeric(
    df_check['TRANSACTION_FEES'].str.strip().str.replace(r'[^\d.\-]','',regex=True),
    errors='coerce'
)

print("── Statistiques TRANSACTION_AMOUNT (unités brutes) ──")
print(df_check['AMOUNT_XOF'].describe().apply(lambda x: f'{x:,.0f}').to_string())


In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(16, 10))
fig.suptitle('Analyse des montants de transactions (XOF)', fontsize=14, fontweight='bold')

amounts = df_check['AMOUNT_XOF'].dropna()
amounts_m = amounts / 1e6   # en millions XOF

# Distribution log
axes[0,0].hist(np.log1p(amounts), bins=80, color='#3498db', edgecolor='none')
axes[0,0].set_title('Distribution log(montant)')
axes[0,0].set_xlabel('log(1 + montant brut)')
axes[0,0].set_ylabel('Fréquence')

# Distribution en millions XOF (99e percentile)
clip_val = amounts_m.quantile(0.99)
axes[0,1].hist(amounts_m.clip(upper=clip_val), bins=80, color='#2ecc71', edgecolor='none')
axes[0,1].set_title(f'Distribution (≤ {clip_val:.2f} M XOF, 99e pct)')
axes[0,1].set_xlabel('Montant (M XOF)')
axes[0,1].set_ylabel('Fréquence')

# Boxplot par statut
status_ok = df_check.dropna(subset=['TRANSACTION_STATUS','AMOUNT_XOF'])
status_ok['log_amount'] = np.log1p(status_ok['AMOUNT_XOF'])
status_ok.boxplot(column='log_amount', by='TRANSACTION_STATUS', ax=axes[1,0],
                  patch_artist=True)
axes[1,0].set_title('Log(montant) par statut')
axes[1,0].set_xlabel('Statut')
axes[1,0].set_ylabel('log(1 + montant)')
plt.sca(axes[1,0])
plt.title('Log(montant) par statut')

# Montant cumulé par service
svc_amount = (
    df_check.groupby('SERVICE_CODE')['AMOUNT_XOF']
    .sum().sort_values(ascending=False).head(10) / 1e9
)
svc_amount.plot(kind='bar', ax=axes[1,1], color='#e74c3c', edgecolor='white')
axes[1,1].set_title('Volume total par service (Mrd XOF)')
axes[1,1].set_xlabel('Service')
axes[1,1].set_ylabel('Milliards XOF')
axes[1,1].tick_params(axis='x', rotation=30)

plt.tight_layout()
plt.savefig('../reports/figures/01_amounts.png', bbox_inches='tight')
plt.show()


In [ ]:
# Transactions à très hauts montants
HIGH = 1_000_000  # 1M en unités brutes
high_tx = df_check[df_check['AMOUNT_XOF'] > HIGH]
print(f"Transactions > 1 000 000 (unités brutes) : {len(high_tx):,}  ({len(high_tx)/len(df_check)*100:.2f}%)")
print(f"Montant max observé : {df_check['AMOUNT_XOF'].max():,.0f}")
print(f"Montant médian      : {df_check['AMOUNT_XOF'].median():,.0f}")


---
## 5. Analyse Temporelle


In [ ]:
# Parse dates
df_check['TX_DATE'] = pd.to_datetime(
    df_check['TRANSACTION_DATE'].str.strip(),
    format='mixed', dayfirst=True, errors='coerce'
)
df_valid = df_check.dropna(subset=['TX_DATE']).copy()
df_valid['hour']  = df_valid['TX_DATE'].dt.hour
df_valid['dow']   = df_valid['TX_DATE'].dt.dayofweek
df_valid['month'] = df_valid['TX_DATE'].dt.month
df_valid['date']  = df_valid['TX_DATE'].dt.date

print(f"Période couverte : {df_valid['TX_DATE'].min()}  →  {df_valid['TX_DATE'].max()}")
print(f"Jours actifs     : {df_valid['date'].nunique():,}")
print(f"Transactions/jour (médiane) : {df_valid.groupby('date').size().median():.0f}")


In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(16, 10))
fig.suptitle('Analyse temporelle des transactions', fontsize=14, fontweight='bold')

day_labels = ['Lun','Mar','Mer','Jeu','Ven','Sam','Dim']
month_labels = ['Jan','Fév','Mar','Avr','Mai','Jun','Jul','Aoû','Sep','Oct','Nov','Déc']

# Volume horaire
hour_cnt = df_valid['hour'].value_counts().sort_index()
night_color = ['#e74c3c' if h >= 22 or h < 6 else '#3498db' for h in hour_cnt.index]
axes[0,0].bar(hour_cnt.index, hour_cnt.values, color=night_color)
axes[0,0].set_title('Volume par heure (rouge = nuit 22h-6h)')
axes[0,0].set_xlabel('Heure de la journée')
axes[0,0].set_ylabel('Transactions')
axes[0,0].set_xticks(range(0, 24))

# Volume par jour de semaine
dow_cnt = df_valid['dow'].value_counts().sort_index()
colors_dow = ['#e74c3c' if i >= 5 else '#2c3e50' for i in range(7)]
axes[0,1].bar([day_labels[i] for i in dow_cnt.index], dow_cnt.values,
               color=[colors_dow[i] for i in dow_cnt.index])
axes[0,1].set_title('Volume par jour de semaine (rouge = week-end)')
axes[0,1].set_xlabel('Jour')
axes[0,1].set_ylabel('Transactions')

# Volume journalier (série temporelle)
daily = df_valid.groupby('date').size()
axes[1,0].plot(range(len(daily)), daily.values, color='#2980b9', linewidth=1)
axes[1,0].fill_between(range(len(daily)), daily.values, alpha=0.2, color='#2980b9')
axes[1,0].set_title('Volume journalier')
axes[1,0].set_xlabel('Jours')
axes[1,0].set_ylabel('Transactions/jour')
axes[1,0].axhline(daily.mean(), color='#e74c3c', linestyle='--', linewidth=1,
                   label=f'Moyenne: {daily.mean():.0f}')
axes[1,0].legend()

# Volume mensuel
month_cnt = df_valid['month'].value_counts().sort_index()
axes[1,1].bar([month_labels[m-1] for m in month_cnt.index], month_cnt.values,
               color=sns.color_palette('Blues_d', len(month_cnt)))
axes[1,1].set_title('Volume par mois')
axes[1,1].set_xlabel('Mois')
axes[1,1].set_ylabel('Transactions')
axes[1,1].tick_params(axis='x', rotation=30)

plt.tight_layout()
plt.savefig('../reports/figures/01_temporal.png', bbox_inches='tight')
plt.show()

night_pct = ((df_valid['hour'] >= 22) | (df_valid['hour'] < 6)).mean() * 100
wknd_pct  = df_valid['dow'].isin([5,6]).mean() * 100
print(f"\n🌙 Transactions nocturnes (22h-6h)  : {night_pct:.2f}%")
print(f"📅 Transactions week-end             : {wknd_pct:.2f}%")


---
## 6. Analyse des Acteurs
> Qui envoie ? Qui reçoit ? Identifier les comportements extrêmes.


In [ ]:
# Top expéditeurs par volume
top_senders = (
    df_check.groupby('SOURCE_PHONE')
    .agg(nb_tx=('TRANSACTION_CODE','count'),
         total_amount=('AMOUNT_XOF','sum'),
         unique_dest=('DESTINATION_PHONE','nunique'))
    .sort_values('nb_tx', ascending=False)
    .head(20)
    .reset_index()
)

print("Top 20 expéditeurs par nombre de transactions :")
print(top_senders.head(10).to_string(index=False))


In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(18, 5))
fig.suptitle('Profil des acteurs (expéditeurs)', fontsize=14, fontweight='bold')

sender_stats = (
    df_check.groupby('SOURCE_PHONE')
    .agg(nb_tx=('TRANSACTION_CODE','count'),
         total_amount=('AMOUNT_XOF','sum'),
         unique_dest=('DESTINATION_PHONE','nunique'))
)

# Distribution nb transactions par sender
nb_tx_clipped = sender_stats['nb_tx'].clip(upper=sender_stats['nb_tx'].quantile(0.99))
axes[0].hist(nb_tx_clipped, bins=50, color='#8e44ad', edgecolor='none')
axes[0].set_title('Nb transactions par expéditeur')
axes[0].set_xlabel('Nombre de transactions')
axes[0].set_ylabel('Fréquence')
axes[0].axvline(sender_stats['nb_tx'].quantile(0.95), color='#e74c3c',
                linestyle='--', label=f"95e pct: {sender_stats['nb_tx'].quantile(0.95):.0f}")
axes[0].legend()

# Distribution destinataires uniques (fan-out)
dest_clipped = sender_stats['unique_dest'].clip(upper=sender_stats['unique_dest'].quantile(0.99))
axes[1].hist(dest_clipped, bins=40, color='#e67e22', edgecolor='none')
axes[1].set_title('Destinataires uniques par expéditeur')
axes[1].set_xlabel('Nb destinataires distincts')
axes[1].set_ylabel('Fréquence')

# Corrélation nb_tx vs unique_dest
sample_senders = sender_stats[sender_stats['nb_tx'] <= sender_stats['nb_tx'].quantile(0.99)]
axes[2].scatter(sample_senders['nb_tx'], sample_senders['unique_dest'],
                alpha=0.3, s=10, color='#16a085')
axes[2].set_title('nb_tx vs unique_dest (expéditeurs)')
axes[2].set_xlabel('Nombre de transactions')
axes[2].set_ylabel('Destinataires distincts')

plt.tight_layout()
plt.savefig('../reports/figures/01_actors.png', bbox_inches='tight')
plt.show()

print(f"\n── Statistiques expéditeurs ──")
print(f"  Expéditeurs uniques             : {len(sender_stats):,}")
print(f"  Médiane transactions/expéditeur : {sender_stats['nb_tx'].median():.0f}")
print(f"  Max transactions (un seul)      : {sender_stats['nb_tx'].max():,}")
print(f"  Expéditeurs avec > 10 tx        : {(sender_stats['nb_tx'] > 10).sum():,}  ({(sender_stats['nb_tx'] > 10).mean()*100:.1f}%)")
print(f"  Fan-out max (destinataires)     : {sender_stats['unique_dest'].max():,}")


---
## 7. Détection Préliminaire d'Anomalies
> Avant tout modèle ML, des règles simples permettent de repérer des comportements suspects.


In [ ]:
# ── Indicateurs d'anomalies simples ──────────────────────────────────────
df_valid_num = df_valid.copy()
df_valid_num['AMOUNT_XOF'] = pd.to_numeric(
    df_valid_num['TRANSACTION_AMOUNT'].str.strip().str.replace(r'[^\d.\-]','',regex=True),
    errors='coerce'
)

THRESHOLDS = [100_000, 500_000, 1_000_000]

for t in THRESHOLDS:
    lower = t * 0.9
    near = ((df_valid_num['AMOUNT_XOF'] >= lower) & (df_valid_num['AMOUNT_XOF'] < t)).sum()
    print(f"  Près du seuil {t:>10,} (±10%) : {near:>7,}  ({near/len(df_valid_num)*100:.3f}%)")

night_tx = ((df_valid_num['hour'] >= 22) | (df_valid_num['hour'] < 6)).sum()
print(f"\n  Transactions nocturnes (22h-6h) : {night_tx:>7,}  ({night_tx/len(df_valid_num)*100:.2f}%)")

zero_fee = (pd.to_numeric(df_valid_num['TRANSACTION_FEES'].str.replace(r'[^\d.\-]','',regex=True),errors='coerce') == 0).sum()
print(f"  Transactions à frais zéro       : {zero_fee:>7,}  ({zero_fee/len(df_valid_num)*100:.2f}%)")


In [ ]:
# ── Profil des transactions REJECTED vs VALIDATED ─────────────────────────
fig, axes = plt.subplots(1, 2, figsize=(14, 5))
fig.suptitle('Profil VALIDATED vs REJECTED', fontsize=14, fontweight='bold')

for status, color, ax in [('VALIDATED','#27ae60', axes[0]), ('REJECTED','#e74c3c', axes[1])]:
    subset = df_valid_num[df_valid_num['TRANSACTION_STATUS'] == status]['AMOUNT_XOF'].dropna()
    if len(subset) == 0:
        continue
    clipped = np.log1p(subset.clip(upper=subset.quantile(0.99)))
    ax.hist(clipped, bins=60, color=color, edgecolor='none', alpha=0.8)
    ax.set_title(f'Log(montant) — {status} ({len(subset):,} tx)')
    ax.set_xlabel('log(1 + montant)')
    ax.set_ylabel('Fréquence')
    ax.axvline(clipped.mean(), color='black', linestyle='--', linewidth=1.2,
               label=f'Moyenne: {clipped.mean():.2f}')
    ax.legend()

plt.tight_layout()
plt.savefig('../reports/figures/01_status_comparison.png', bbox_inches='tight')
plt.show()


---
## 8. Synthèse & Recommandations Feature Engineering

### Observations clés

| Observation | Impact |
|---|---|
| Transactions nocturnes présentes | → Feature `tx_is_night` |
| Pics de volume sur certaines heures | → Features rolling 1h / 24h |
| Quelques expéditeurs avec très nombreux destinataires | → Feature `unique_dest_7d` (fan-out) |
| Montants proches des seuils 100k / 500k / 1M XOF | → Features `is_near_threshold_*` |
| Transactions à frais zéro sur montants élevés | → Feature `fee_ratio` |
| Certains codes service concentrent le volume | → Encoder `SERVICE_CODE` |

### Variables retenues pour le Feature Engineering
Voir **`02_Feature_Engineering.ipynb`** pour la construction complète des 18 features.

### Prochaine étape
```
02_Feature_Engineering.ipynb
  → 03_Isolation_Forest.ipynb  (scoring anomalie)
  → 04_Graph_Analysis.ipynb    (réseaux suspects)
  → 07_Explainability.ipynb    (SHAP + explications)
```


In [ ]:
print("✅ EDA terminée.")
print(f"   Figures sauvegardées dans : ../reports/figures/")
print(f"   Fichiers créés :")
import os
figs = [f for f in os.listdir('../reports/figures') if f.startswith('01_')]
for f in sorted(figs):
    print(f"     {f}")
